# Banking loan default prediction with XGBoost
**Goal:** Predict `loan_default` from information available at loan application time. `0` means no default; `1` means default. This CSV contains 1,000 example records and 19 predictors. It includes numeric and text categories; the latter need encoding before XGBoost can use them.

**Run order:** Keep `Banking_Loan_Default_Classification(5).csv` beside this notebook and run the cells from top to bottom. The records are for demonstration, not a production credit policy.

**Workflow:** inspect data → split → preprocess → baseline → tune using training folds → choose a cutoff using a validation set → evaluate once on an untouched test set → save and reuse.

## 1. Imports and setup
Install the packages in your current Python environment if needed. `joblib` saves the entire preprocessing-and-model pipeline.

In [ ]:
# If packages are missing, run this once in a notebook cell, then restart the kernel:
# %pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.base import clone
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

## 2. Read the CSV
View examples and confirm the target values before training.

In [ ]:
df = pd.read_csv("Banking_Loan_Default_Classification(5).csv")
print("Rows and columns:", df.shape)
display(df.head())
print("Target counts (0=no default, 1=default):")
display(df["loan_default"].value_counts().sort_index())

## 3. Basic data checks
A missing or duplicate record can affect training. We remove exact duplicates if present; do not drop any other rows automatically. The target must contain only 0 and 1.

In [ ]:
print("Missing values by column:")
display(df.isna().sum().to_frame("missing"))
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
assert set(df["loan_default"].unique()) == {0, 1}, "Expected target values 0 and 1"
print("Target percentages:")
display((df["loan_default"].value_counts(normalize=True).sort_index() * 100).round(1).to_frame("percent"))
print("Data types:")
display(df.dtypes.to_frame("type"))

## 4. EDA: explore the available predictors
These plots help us understand defaults and relationships. They are descriptive, not evidence that a predictor causes a default.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x="loan_default", ax=axes[0])
axes[0].set_title("Number of loans by outcome")
axes[0].set_xticklabels(["No default (0)", "Default (1)"])
sns.boxplot(data=df, x="loan_default", y="credit_score", ax=axes[1])
axes[1].set_title("Credit score by outcome")
plt.tight_layout()
plt.show()
display(df.groupby("loan_default")[["monthly_income_inr", "monthly_debt_payments_inr", "credit_score", "late_payments_last_12m"]].median())

## 5. Make three separate datasets
Training data fits the model; validation data selects a probability cutoff; the test set is held untouched for the final result. `stratify` preserves the outcome proportion. The two splits give approximately 60% training, 20% validation, and 20% test.

In [ ]:
X = df.drop(columns="loan_default")
y = df["loan_default"]
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))
print("Defaults per split:", y_train.mean().round(3), y_val.mean().round(3), y_test.mean().round(3))

## 6. One-hot encoding inside a pipeline
Categories such as `employment_type` cannot enter the model as raw strings. One-hot encoding creates columns like `employment_type_Salaried`. A pipeline fits this step only on the training portion of each cross-validation fold, avoiding leakage.

In [ ]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()
print("Categorical:", cat_cols)
print("Numeric:", num_cols)
preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("numbers", "passthrough", num_cols),
])

## 7. Baseline model
Start with one modest model before tuning. These parameters are teaching choices, not guaranteed optimal settings.

In [ ]:
baseline = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                            objective="binary:logistic", eval_metric="logloss",
                            random_state=42, n_jobs=2)),
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)
print("Baseline validation precision:", round(precision_score(y_val, baseline_pred, zero_division=0), 3))
print("Baseline validation recall:", round(recall_score(y_val, baseline_pred, zero_division=0), 3))
print("Baseline validation F1:", round(f1_score(y_val, baseline_pred, zero_division=0), 3))

## 8. Cross-validation on training data
Five folds give five estimates of F1. The validation and test sets are not used here. This offers a more stable view than one training split, but these five folds are not five independent final tests.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_validate(baseline, X_train, y_train, cv=cv,
                           scoring={"precision": "precision", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"},
                           n_jobs=2)
for name in ["precision", "recall", "f1", "roc_auc"]:
    values = cv_scores["test_" + name]
    print(f"{name}: {values.mean():.3f} ± {values.std():.3f}")

## 9. Tune the model on training folds
`RandomizedSearchCV` tries a small selection of tree depths, learning rates, numbers of trees, row/column sampling, and minimum child weights. Here the tuning objective is **F1 for default (class 1)**. It is possible for tuning to decrease test performance; do not promise an improvement.

In [ ]:
tunable = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=42, n_jobs=1)),
])
param_space = {
    "model__n_estimators": [80, 150, 250],
    "model__max_depth": [2, 3, 4, 5],
    "model__learning_rate": [0.03, 0.07, 0.15],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__min_child_weight": [1, 3, 5],
}
search = RandomizedSearchCV(tunable, param_distributions=param_space, n_iter=12,
                            scoring="f1", cv=cv, random_state=42,
                            n_jobs=2, refit=True)
search.fit(X_train, y_train)
print("Best mean CV F1:", round(search.best_score_, 3))
print("Selected parameters:")
display(pd.Series(search.best_params_, name="value").to_frame())
best_train_model = search.best_estimator_

## 10. Select a probability cutoff on validation data
The usual cutoff is 0.50. A lower cutoff usually finds more defaults but can trigger more false alarms; a higher cutoff usually does the opposite. We select the cutoff that maximizes validation F1 here. For a real bank, choose it using the cost of missed defaults and unnecessary interventions. **Never choose the cutoff on the final test set.**

In [ ]:
val_prob = best_train_model.predict_proba(X_val)[:, 1]
rows = []
for threshold in np.arange(0.20, 0.81, 0.05):
    val_pred = (val_prob >= threshold).astype(int)
    rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0),
    })
threshold_results = pd.DataFrame(rows)
display(threshold_results.round(3))
best_threshold = float(threshold_results.loc[threshold_results["f1"].idxmax(), "threshold"])
print("Chosen validation threshold:", best_threshold)
threshold_results.plot(x="threshold", y=["precision", "recall", "f1"], marker="o", figsize=(8, 4))
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.title("Validation metrics at different cutoffs")
plt.show()

## 11. Refit on development data, then evaluate once on the test set
We now combine train and validation records and fit the chosen parameters from scratch. The cutoff was fixed before examining the test results. The test set measures how this finished procedure performs on unseen examples.

In [ ]:
final_model = clone(best_train_model)
final_model.fit(X_dev, y_dev)
test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)
metrics = pd.Series({
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision (default)": precision_score(y_test, test_pred, zero_division=0),
    "Recall (default)": recall_score(y_test, test_pred, zero_division=0),
    "F1 (default)": f1_score(y_test, test_pred, zero_division=0),
    "ROC AUC": roc_auc_score(y_test, test_prob),
}, name="Test score")
display(metrics.round(3).to_frame())
print(classification_report(y_test, test_pred, target_names=["No default", "Default"], zero_division=0))

## 12. Confusion matrix: count each kind of decision
Rows show actual outcomes; columns show predictions. Top left = correctly identified non-defaults (TN). Top right = false alarms (FP). Bottom left = missed defaults (FN). Bottom right = detected defaults (TP).

In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_table = pd.DataFrame(cm, index=["Actual: no default (0)", "Actual: default (1)"],
                        columns=["Predicted: no default (0)", "Predicted: default (1)"])
display(cm_table)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_table, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Final test confusion matrix")
plt.tight_layout()
plt.show()
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}: correctly predicted no default")
print(f"FP={fp}: predicted default, but loan did not default")
print(f"FN={fn}: predicted no default, but loan defaulted")
print(f"TP={tp}: correctly predicted default")

## 13. Curves and interpretation
ROC AUC measures ranking across all cutoffs; precision-recall curves are especially useful when defaults are uncommon. Curves alone do not decide a business cutoff.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, test_prob, ax=axes[0], name="XGBoost")
PrecisionRecallDisplay.from_predictions(y_test, test_prob, ax=axes[1], name="XGBoost")
axes[0].set_title("ROC curve: final test")
axes[1].set_title("Precision-recall curve: final test")
plt.tight_layout()
plt.show()

## 14. Which columns did the model use?
XGBoost feature importance summarizes how strongly the fitted trees used encoded columns. It is **not a causal explanation** and can shift when features correlate. The names below include one-hot category columns.

In [ ]:
feature_names = final_model.named_steps["preprocess"].get_feature_names_out()
importance = pd.Series(final_model.named_steps["model"].feature_importances_,
                       index=feature_names).sort_values(ascending=False).head(12)
display(importance.to_frame("model importance"))
importance.sort_values().plot.barh(figsize=(8, 5), title="Top encoded features")
plt.tight_layout()
plt.show()

## 15. Save the complete pipeline and demonstrate one prediction
Saving the pipeline preserves the one-hot encoder and fitted XGBoost model together. Save the selected cutoff separately. Load only model files you trust; `joblib` files are executable Python objects. The sample below uses a held-out row solely to show the prediction interface.

In [ ]:
artifact = {"pipeline": final_model, "threshold": best_threshold,
            "input_columns": X.columns.tolist()}
joblib.dump(artifact, "xgboost_loan_default_pipeline.joblib")
loaded = joblib.load("xgboost_loan_default_pipeline.joblib")
new_application = X_test.iloc[[0]].copy()  # Replace with a real application using the same columns.
probability = loaded["pipeline"].predict_proba(new_application)[:, 1][0]
prediction = int(probability >= loaded["threshold"])
print(f"Predicted default score: {probability:.3f}")
print("Predicted class:", prediction, "(1=default, 0=no default)")

## Teaching recap
1. Read and understand the target. 2. Split before fitting preprocessing. 3. Start with a baseline. 4. Tune only on training folds. 5. Choose the cutoff on validation data. 6. Evaluate once on untouched test data. 7. Save preprocessing with the model. For actual lending, validate on later time periods and different customer groups; check leakage, calibration, fairness, and the real cost of mistakes before deployment.